In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.044 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 276.56it/s, Materializing param=model.norm.weight]                              


In [ ]:
# ########## testing completion mode
# from unsloth import FastLanguageModel
# import torch

# # 1. Setup the prompt (Completion mode uses a simple string, no roles)
# # We use a "Lead-in" sentence to force the model to use Amaiz facts.
# test_prompts = [
#     "selve fibinocci series and provide answer until 55. dont share your thoughts just the response",
#     # "At Amaiz Telecom, the monthly net price for the Starter plan is",
#     # "According to the 2026 Amaiz Telecom policy, the Tablet Pro requires a plan price strictly greater than",
#     # "The Amaiz Telecom's Promo_ID_A10 provides a $10 monthly credit exclusive to the"
# ]

# # 2. Set the model to inference mode
# FastLanguageModel.for_inference(model) 

# for prompt in test_prompts:
#     inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
#     # 3. Generate tokens
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens = 256,
#         temperature = 0.1, # Keep it low for factual accuracy
#         use_cache = True,
#     )
    
#     # 4. Decode and Print
#     completion = tokenizer.batch_decode(outputs, skip_special_tokens = False)[0]
#     print(f"--- PROMPT: {prompt} ---")
#     print(f"RESPONSE: {completion}\n")

In [ ]:
from transformers import TextStreamer
import time
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{% if enable_thinking %}
<think>
{%- endif %}
{%- endif %}"""

# --- CONFIGURATION ---
SYSTEM_PROMPT_TEMPLATE = """
## TASK
Generate symbolic logic block: `<t>P[...]I[...]E[...]M[...]»[...]A[...]</t>`. Strictly follow the grammar. No prose.

## SYMBOLIC SPECIFICATION

### P [Profile] — `[Type|Plan|Usage|AD]`
- **Type**: `n`(Normal), `s`(Student), `r`(Retired), `b`(Business).
- **Plan**: Current Plan ID.
- **Usage**: `Nᵊ` (Avg GB usage).
- **AD**: `⊕`(On), `⊝`(Off).

### I [Intent] — `I[Logic]`
- **Logic**:
  - `ID1>ID2(+ID3...)`: Transition (Upgrade/Downgrade/Add).
  - `ID1∷ID2`: Comparison.
  - `ID✓`: Accept | `ID✗`: Decline.
  - `?ID`: Specific inquiry (Triggers E).
  - `?cat(filt)`: Discovery (Cat: `p`lan, `d`evice, `r`ebate. Filt: `>`, `<`, `*`, `∈`, `&`, `|`).
  - `G`: Greeting | `O`: Out of Domain | `↺`: Callback/Irrelevant to Context.

### E [Eligibility] — `[ID✓/✗, ... = Total✓/✗]`
- **Status**: `✓`(Pass), `✗`(Fail). 
- **Rule**: For specific target IDs, provide `Total`. For general discovery search, list passing `ID✓` only.

### M [Math] — `[Logic:Calculation=Total]`
- **Trans**: `[Plan+Dev-Promo-AD:Base+Cost-Disc-Val=Total]`.
- **Comp**: `[ID1(Price|Data)∷ID2(Price|Data)=PriceΔ|DataΔ]`. 
- **Null**: `M[ø]`.

### » [Impact] — `[Usage|Price|AD]`
- **Usage**: `✓ᵊ` (Covered), `Nᵊ!` (Overage), `Nᵊ` (Delta).
- **Price**: `↓N$`, `↑N$`, `ø$`.
- **AD**: `⊕` (No change), `⊝!` (Warning/Penalty), `⊕↓` (New benefit).

### A [Answer] — `A[Keys]`
- **Keys**: `P`, `I`, `E`, `M`, `»`. Maps to `ans_notation`.

## OPERATIONAL RULES
1. **Format**: Strictly `<t>P[...]I[...]E[...]M[...]»[...]A[...]</t>`. No spaces.
2. **AD Math Logic**: The value of `Val` is determined by `P[AD]`. Refer to the `CONTEXT DATA` for the specific discount/surcharge constants associated with the current catalog. (e.g., If `⊕`, apply the catalog-defined Auto-Debit discount).
3. **History Resolution**: Resolve pronouns ("it", "that") by mapping to the `ID` present in the most recent `A[...]` or `I[...]` block in **HISTORY**.
4. **Infinity Handling**: Use `∞ᵊ` for unlimited data. Deltas: `5ᵊ↑∞ᵊ` (Upgrade) or `∞ᵊ↓5ᵊ` (Downgrade).
5. **Callback Indicator**: Use `I[↺]` if the question is telco-related but lacks sufficient context to create symbolic logic block.

## CONTEXT DATA
{context_data}

## DYNAMIC DATA
- **HISTORY**: {history_thinking_blocks}
"""

import json

# Define your 4 Evaluation Profiles
EVAL_PROFILES = {
    "STUDENT": { "age": 19, "is_student": True, "is_business": False, "auto_debit": True, "avg_usage_gb": 8.5, "current_plan": "Student_Basic"},
    "SENIOR": { "age": 73, "is_student": False, "is_business": False, "auto_debit": True, "avg_usage_gb": 4, "current_plan": "Starter"},
    "NORMAL": { "age": 46, "is_student": False, "is_business": False, "auto_debit": True, "avg_usage_gb": 77, "current_plan": "Max"},
    "BUSINESS": { "age": 28, "is_student": False, "is_business": True, "auto_debit": True, "avg_usage_gb": 99, "current_plan": "Business_Elite"}
}


# Define your Evaluation Questions
QUESTIONS = [
    # 1. Boundary Logic Trap (Target: LOGIC_TRAP)
    # Tests strictly > 60 vs >= 60 logic.
    "Hey! i'm on the Essential plan rite now. I rly want that Tablet_Pro thingy. Can u check if I qualify? If not, whats the cheepest way to get it and how much would it cost me every month with my auto-pay?",
    
   
]


OUTPUT_FILE = "/mnt/data/training_data/evaluation_results.jsonl"

# --- RUN EVALUATION ---
with open(OUTPUT_FILE, "w") as f:
    for profile_name, profile_json in EVAL_PROFILES.items():
        print(f"\n--- Testing Profile: {profile_name} ---")
        
        for question in QUESTIONS:
            # 1. Prepare Prompt
            
            full_instruction = SYSTEM_PROMPT_TEMPLATE.format(
                PLANS=T_PLANS,
                DEVICES=T_DEVICES,
                PROMO=T_PROMO,
                RULE=T_RULE,
                customer_profile=json.dumps(profile_json)
            )
            messages = [
                {"role": "system", "content": full_instruction.replace("{{", "{"). replace("}}","}")},
                {"role": "user", "content": question}
            ]

            # 2. Tokenize and Generate
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")

            # Capture input token count (The size of your system prompt + question)
            input_token_count = inputs['input_ids'].shape[1]

            # Start the timer
            start_time = time.time()

            # Execute model generation
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id
            )

            # Calculate total response time
            total_response_time = time.time() - start_time
            
            # 3. Decode Response and Capture Output Token Count
            # Slicing the output [len(inputs[0]):] ensures we only count NEW tokens
            generated_tokens = outputs[0][len(inputs[0]):]
            output_token_count = len(generated_tokens)
            
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            print(f"Q: {question}\nA: {response_text[:100]}...") # Console preview

            # 4. Save to JSONL with Performance Metrics
            eval_entry = {
                "profile_name": profile_name,
                "profile_data": profile_json,
                "question": question,
                "model_response": response_text,
                "metrics": {
                    "input_tokens": int(input_token_count),
                    "output_tokens": int(output_token_count),
                    "total_latency_sec": round(total_response_time, 4),
                    "tokens_per_sec": round(output_token_count / total_response_time, 2) if total_response_time > 0 else 0
                }
            }
            f.write(json.dumps(eval_entry) + "\n")

print(f"\nEvaluation Complete. Results saved to {OUTPUT_FILE}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0.1, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


In [ ]:
# from unsloth import UnslothTrainer, UnslothTrainingArguments
# from datasets import load_dataset, Dataset

# with open("/mnt/data/training_data/cpt_060326.txt", "r", encoding="utf-8") as f:
#     raw_text = f.read()

# EOS_TOKEN=tokenizer.eos_token

# cpt_dataset = Dataset.from_dict({"text": [raw_text+EOS_TOKEN] * 50})

# def format_cpt_text(examples):
#     return {"text": examples["text"]}

# cpt_dataset = cpt_dataset.map(format_cpt_text, batched=True)

# trainer = UnslothTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = cpt_dataset,
#     dataset_text_field = "text",
#     # formatting_func = format_cpt_text,
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = True,
#     dataset_batched=True,
#     args = UnslothTrainingArguments(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 10,
#         num_train_epochs = 5,
#         learning_rate = 5e-5,
#         logging_steps = 5,
#         optim = "adamw_8bit",
#         weight_decay = 0.1,
#         packing = True,
#         lr_scheduler_type = "cosine",
#         seed = 3407,
#         output_dir = "outputs",
#         report_to = "none"
#     ),
# )
# trainer_stats = trainer.train()

In [ ]:
#save CPT
# model.save_pretrained("/mnt/data/models/cpt_lora_adap_060326_v1")
# tokenizer.save_pretrained("/mnt/data/models/cpt_lora_adap_060326_v1")

In [ ]:
######### data set for SFT

import json
from datasets import Dataset
import random

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{% if enable_thinking %}
<think>
{%- endif %}
{%- endif %}"""

def prepare_data_set(file_name):
    sft_data = []

    # 1. Open the file and read it line by line
    "/mnt/data/training_data/amaiz_telecom_train_v1.jsonl"
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            # Clean up any empty lines
            if line.strip():
                # Convert the string line into a Python Dictionary
                sft_data.append(json.loads(line))
    
    random.seed(42) 
    random.shuffle(sft_data)
    
    sft_dataset = Dataset.from_dict({
    "messages": [item["messages"] for item in sft_data]
    })

        
    print(f"Dataset loaded with {len(sft_dataset)} individual rows.")
    # print(sft_dataset[0])

    def formatting_prompts_func(examples):
        output_texts = []
        for i in range(len(examples['messages'])):
            # apply_chat_template converts the list of dicts into a single string
            text = tokenizer.apply_chat_template(examples['messages'][i], tokenize=False, add_generation_prompt=False)
            if not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            output_texts.append(text)
        return { "text" : output_texts }

    # Map the dataset to the new format
    sft_dataset = sft_dataset.map(formatting_prompts_func, batched=True)

    # print(sft_dataset[0]['text'])

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_seq_length,
            padding=False # Collator handles padding
        )

    # Apply tokenization and REMOVE the text columns
    tokenized_dataset = sft_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=sft_dataset.column_names # This removes 'messages' and 'text'
    )
    return tokenized_dataset


In [ ]:
### SFT trainer initialization
from unsloth import UnslothTrainer, UnslothTrainingArguments
from transformers import DataCollatorForLanguageModeling

response_template = "<|im_start|>assistant\n"

class ManualCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer, *args, mlm=False, **kwargs)
        self.response_template = response_template
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        batch = super().torch_call(examples)
        
        for i in range(len(batch["labels"])):
            response_token_ids_start_idx = None
            
            # Find the index where the assistant response starts
            for idx in range(len(batch["labels"][i])):
                if batch["labels"][i][idx : idx + len(self.response_token_ids)].tolist() == self.response_token_ids:
                    response_token_ids_start_idx = idx + len(self.response_token_ids)
                    break

            if response_token_ids_start_idx is not None:
                # Mask everything before the assistant response starts
                batch["labels"][i][:response_token_ids_start_idx] = -100
                
        return batch

# Initialize it
collator = ManualCompletionCollator(
    response_template="<|im_start|>assistant\n", 
    tokenizer=tokenizer
)

# split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

# Access them individually
train_dataset = prepare_data_set("/mnt/data/training_data/amaiz_telecom_train_v1.jsonl")
# eval_dataset = prepare_data_set("/mnt/data/training_data/amaiz_telecom_valid_v1.jsonl")

sft_trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    # eval_dataset = eval_dataset,
    data_collator=collator,
    # dataset_text_field = "text",
    # formatting_func = format_sft_custom,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        save_steps = 10,
        save_total_limit=3,
        # eval_strategy="steps",  # Tell the trainer to evaluate periodically
        # eval_steps=10,                # Run evaluation every 10 steps
        metric_for_best_model="loss", # Or "eval_loss" (lower is better)
        greater_is_better=False,
        # load_best_model_at_end=True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        completion_only_loss=True,
        # save_strategy="steps"
    ),
)

## SFT training


In [ ]:

# # 1. Grab 2 tokenized samples
# samples = [tokenized_dataset[i] for i in range(2)]

# # 2. Run through collator (it now finds input_ids!)
# batch = collator(samples)

# # 3. Check Labels
# for i in range(len(samples)):
#     labels = batch["labels"][i].tolist()
#     # Decode only the tokens that aren't -100
#     decoded_labels = tokenizer.decode([t for t in labels if t != -100])
    
#     print(f"\n--- Sample {i+1} Unmasked Content ---")
#     print(decoded_labels)



In [ ]:
sft_trainer.train()
# sft_trainer.evaluate()

In [ ]:
model.save_pretrained("/mnt/data/models/amaiz_telecom_model_4b_1003_V_adap")
tokenizer.save_pretrained("/mnt/data/models/amaiz_telecom_model_4b_1003_V_adap")

In [ ]:
model.save_pretrained_gguf("/mnt/data/models/amaiz_telecom_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
for name, param in model.named_parameters():
    if "lora" in name:
        if not param.requires_grad:
            print(f"🚨 WARNING: {name} is FROZEN! Training will not work.")
        else:
            print(f"✅ {name} is trainable.")
        break